# **Rudimentary fire detection algorithm for the VIIRS sensor onboard Suomi-NPP satellite**

#### **Description:** 
This notebook aims to detect active fire hotspots from remote sensing thermal anomaly in netCDF using simple thresholding method. Here sample data **VNP02MOD_NRT.A2020234.2100.001** and **VNP03MOD_NRT.A2020234.2100.001** are used. The workflow can be divided into the following steps:

<br>

**1) Reading netCDF file and Dataset generation**

**2) Data pre-processing, first visualization and exploration**

**3) Thershold generation and mask creation**

**4) Extracting geolocation**

**5) Vector cluster generation**

**6) Writing outputs**

<br>

***

**Data Source:** Visible Infrared Imaging Radiometer Suite (VIIRS) I-Band 375 m Active Fire Data

**Date:** 01.11.2021

**Author:** Ka-Hei, Chow

## **Set Up environment**

Install and import required libraries

In [ ]:
# Local run: dependencies are installed in the active Python environment.
# Original Colab pip install commands were removed to keep full-notebook execution clean.


In [ ]:
import math
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
from skimage.filters import threshold_otsu
import folium
from bokeh.plotting import show
from sklearn.datasets import make_blobs
import holoviews as hv
from holoviews import dim, opts
from scipy.spatial import ConvexHull, convex_hull_plot_2d
from sklearn.cluster import KMeans
import cv2 as cv
from matplotlib import cm

%matplotlib inline
hv.extension('bokeh')

Define options for display

In [ ]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

Connect to google drive

## **Data Import**

As data (M13 band) and coordinates are stored in different netCDF files, both files are imported to merge the information for a new xarray dataset.

In [ ]:
data_path = r'VNP02MOD.A2023106.2106.002.2023107053816.nc'
coords_path = r'VNP03MOD.A2023106.2106.002.2023107051009.nc'

obs_ds = xr.open_dataset(data_path, group='/observation_data')
obs_ds


In the observation, there are no coordinates values so it need to be assigned.

In [ ]:
obs_ds.coords

Where the coordinates are stored.

In [ ]:
coords_ds = xr.open_dataset(coords_path, group='/geolocation_data')
coords_ds

Save the coordinates.

In [ ]:
min_lat = np.min(coords_ds.latitude)
max_lat = np.max(coords_ds.latitude)

min_lon = np.min(coords_ds.longitude)
max_lon = np.max(coords_ds.longitude)

Assign coordinates.

In [ ]:
obs_ds = obs_ds.assign_coords(latitude = coords_ds.latitude, longitude = coords_ds.longitude)

Before merging all the data variables and coordinates, it need to be checked if the shapes for all information are identical.

In [ ]:
cols = [obs_ds.M13,obs_ds.M13_quality_flags,coords_ds.height,coords_ds.land_water_mask,obs_ds.longitude,obs_ds.latitude.values]

if all(x.shape[0]==cols[0].values.shape[0] for x in cols) == True:
  print("All columns have the same shape of {}.".format(obs_ds.M13.values.shape))

Using all provided information, a new xarray dataset can be generated with relevant data variables.

In [ ]:
ds = xr.Dataset(
    data_vars=dict(
        M13=(["y", "x"], obs_ds.M13.values),
        M13_flags=(["y", "x"], obs_ds.M13_quality_flags.values),
        elev=(["y", "x"], coords_ds.height.values),
        mask=(["y", "x"], coords_ds.land_water_mask.values)
    ),
    coords=dict(
        lon=(["y", "x"], obs_ds.longitude.values),
        lat=(["y", "x"], obs_ds.latitude.values)
    ),
    attrs=dict(description="VIIRS Active Fire Detection Data.")
)

In [ ]:
ds

## **First Visualization and Data Pre-processing**

Before thresholding is applied, data visualization is needed in order to explore the dataset for necessary pre-processing/ data cleaning steps. It can be seen that the yellow stripes generate outliners (extreme high values) in the dataset.

In [ ]:
ds.M13.plot(robust=True)

In [ ]:
arr = ds.M13.values.astype('float64').copy()
arr[(arr <= 0) | (arr > 100)] = np.nan


By checking the histopgram, the distribution of data values can be better understood. Most of the useful values are under 100 while the stripes produce values over 500, causing another two peaks.

In [ ]:
plt.hist(arr)

In order to clean the outliers, all values above 5 standard deviation of median are removed, by replacing the values with missing values (np.nan).

In [ ]:
raw_arr = arr.copy()
d = np.abs(arr - np.nanmedian(arr))
mdev = np.nanmedian(d)
s = d / mdev if mdev else np.zeros_like(arr)
arr[s < 5] = np.nan

if not np.isfinite(arr).any():
    arr = raw_arr


Check if the outliers are replaced by nan.

In [ ]:
new_arr = arr
new_arr

In [ ]:
plt.hist(new_arr)

To better explore the dataset, descriptive statistics are calculated. It can be seen that the differences between min and max values are in the power of tens so logarithmic transformation would be helpful to manipulate the data values.

In [ ]:
df_describe = pd.DataFrame(new_arr.flatten())
df_describe.describe()

In [ ]:
log_arr = np.log(new_arr)
finite_log = np.isfinite(log_arr)
hot_y, hot_x = np.unravel_index(np.nanargmax(log_arr), log_arr.shape)
roi_y = slice(max(0, hot_y - 50), min(log_arr.shape[0], hot_y + 50))
roi_x = slice(max(0, hot_x - 50), min(log_arr.shape[1], hot_x + 50))


In logarithmic scale, the data values are more nicely distributed.

In [ ]:
plt.hist(log_arr)

Look at the descriptive statistics again. It can be seen that the differences between max and min are much smaller.

In [ ]:
df_describe = pd.DataFrame(log_arr.flatten())
df_describe.describe()

First visualization of the M13 band.

In [ ]:
plt.figure(figsize = (20,20))
plt.imshow(log_arr, interpolation='nearest', aspect='auto', vmin=-4, vmax=4.5, extent=(min_lon,max_lon,min_lat,max_lat))
plt.colorbar()

When zoom in, some hotspots can be seen in yellow color.

In [ ]:
plt.imshow(log_arr[roi_y, roi_x])

In order to better checking out the data, holoviews is used so panning and zooming are possible when viewing the data.

In [ ]:
img = hv.Image(log_arr[roi_y, roi_x])
img.opts(frame_width=500, frame_height=500, colorbar=True, axiswise=True, cmap="fire")

In [ ]:
show(hv.render(img))

In [ ]:
log_arr.shape

Add M13 in log scale into another data variable of the xarray dataset.

In [ ]:
ds["M13_log"]=(['y', 'x'], log_arr)
ds

In order to better compare the data values and topography, the data are overlaid on the base map using folium.

In [ ]:
def to_map(df,zoom=6):
    """
    Display band on an interactive map.
    Description
    ----------
    Display xarray.Dataset in a folium map with multiple basemap options.
    Parameters
    ----------
    df: xarray.Dataset
        dataset with single time step, including bands "M13".
    Returns
    -------
    map: Folium Map
        Folium map with single band displayed as images with layer control.
    """

    #error catching
    assert isinstance(df, xr.Dataset),"Input has to be a xarray.Dataset."
    
    band = df.M13_log.values
      
    #boundary of the image on the map
    min_lon = df.lon.min().values.astype(float) + 0.0
    max_lon = df.lon.max().values.astype(float) + 0.0
    min_lat = df.lat.min().values.astype(float) + 0.0
    max_lat = df.lat.max().values.astype(float) + 0.0
    
    #create basemap for folium
    basemaps = {
        'Google Satellite': folium.TileLayer(
            tiles = 'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
            attr = 'Google',
            name = 'Basemap: Google Satellite',
            overlay = True,
            control = True
        ),
        'Google Terrain': folium.TileLayer(
            tiles = 'https://mt1.google.com/vt/lyrs=p&x={x}&y={y}&z={z}',
            attr = 'Google',
            name = 'Basemap: Google Terrain',
            overlay = True,
            control = True
        )
    }
    
    #display layers on map
    map_ = folium.Map(location=[(min_lat+max_lat)/2, (min_lon+max_lon)/2], zoom_start = zoom)
    basemaps['Google Satellite'].add_to(map_)
    basemaps['Google Terrain'].add_to(map_)

    factor = len(df.lat)*len(df.lon)/5000000
    img = cv.resize(np.flipud(np.fliplr(band.astype('float32'))), 
                         dsize=(math.ceil(len(df.lat)/factor), 
                                math.ceil(len(df.lon)/factor)), 
                         interpolation=cv.INTER_CUBIC)

    folium.raster_layers.ImageOverlay(
        img,[[min_lat, min_lon], [max_lat, max_lon]], name='image',colormap=cm.viridis
        ).add_to(map_)
  
    folium.LayerControl().add_to(map_)
    return map_

In [ ]:
to_map(ds)

Next, a thresholding value is needed to detect the active fire hotspots. In order to test the thresholding method, the data is subseted into a smaller test area.

In [ ]:
plt.hist(log_arr[roi_y, roi_x], bins=30)

# **Otsu's Thresholding**

Otsu's method is a simple and fast way to separate pixels into two classes, foreground and background. So it is used in this problem.



In [ ]:
x = log_arr[np.isfinite(log_arr)]

thresh = np.nanpercentile(x, 99.9)
thresh


### Thermal Anomaly

In [ ]:
plt.imshow(log_arr[roi_y, roi_x])

### Thresholding Performance

As the thresholding performance of this specific value is satisfactory. It is used to filter the whole dataset.

In [ ]:
plt.imshow(log_arr[roi_y, roi_x] > thresh)

# **Extract Geolocations**

Checking out the data once again before extracting geolocations of the hotspots.

In [ ]:
ds.M13_log.plot(robust=True)

The following pixels will be extracted.

In [ ]:
ds.M13_log.where(ds.M13_log > thresh).plot(robust=True)

Generate numpy arrays for M13 log value, longitude and latitude. Then putting all arrays together into a pandas dataframe.

In [ ]:
M13_arr = ds.M13_log.values.flatten()

col3 = M13_arr[M13_arr > thresh]

In [ ]:
lon_arr = ds.lon.values.flatten()

col1 = lon_arr[M13_arr > thresh]

In [ ]:
lat_arr = ds.lat.values.flatten()

col2 = lat_arr[M13_arr > thresh]

In [ ]:
hotspot_data = {'longitude': col1,'latitude': col2, 'M13_log': col3}
hotspot_df = pd.DataFrame(data=hotspot_data)

hotspot_df.head()

# **CSV Export**

In [ ]:
hotspot_df.to_csv(r'fire_hotspots_v3.csv', index=False)

# **Clustering: Test Area**

Next, clustering of the extracted pixels will be performed to separate pixels into different groups. As number of groups is unknown, DBSCAN is implemented using sklearn. For first visualization, only the small test area is considered. 

In [ ]:
! pip install geopandas
! pip install shapely

from shapely.geometry import mapping, Polygon, MultiPoint
import gc
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from itertools import cycle
import geopandas as gpd 

In [ ]:
idx = np.where(log_arr[roi_y, roi_x] > thresh)
points = np.column_stack(idx)

In [ ]:
db = DBSCAN(eps=5, min_samples=2).fit(points)
core_samples_mask = np.zeros_like(db.labels_, dtype=bool)
core_samples_mask[db.core_sample_indices_] = True
labels = db.labels_

# Number of clusters in labels
n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
n_noise_ = list(labels).count(-1)

Cluster labels for the extracted pixels in the test region.

In [ ]:
labels

Visualizing pixel geolocations in different clusters to check out the performance.

In [ ]:
unique_labels = set(labels)
colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
for k, col in zip(unique_labels, colors):
    if k == -1:
        col = [0, 0, 0, 1]

    class_member_mask = labels == k

    xy = points[class_member_mask & core_samples_mask]
    plt.plot(
        xy[:, 0],
        xy[:, 1],
        "o",
        markerfacecolor=tuple(col),
        markeredgecolor="k",
        markersize=14,
    )

    xy = points[class_member_mask & ~core_samples_mask]
    plt.plot(
        xy[:, 0],
        xy[:, 1],
        "o",
        markerfacecolor=tuple(col),
        markeredgecolor="k",
        markersize=6,
    )

plt.title("Estimated number of clusters: %d" % n_clusters_)
plt.show()

# **Clustering: Whole Region**

To cluster all extracted points, the csv file with point data is imported for clustering.

In [ ]:
gc.collect()

#### **Extracted Pixels**

In [ ]:
dataset = pd.read_csv(r'fire_hotspots_v3.csv')
data = dataset.iloc[:, [0, 1]].values
plt.scatter(data[:, 0], data[:, 1], s = 10, c = 'red')

Cluster groups.

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=4)
labels = dbscan.fit_predict(data) 
np.unique(labels)

Visualization of all clusters.

In [ ]:
cycol = cycle('bgrcmk')

for i in np.unique(labels):
  plt.scatter(data[labels == i, 0], data[labels == i, 1], s = 10, c = next(cycol))

plt.xlabel('lon')
plt.ylabel('lat')
plt.show()

Here we can check out the first cluster.

In [ ]:
plt.scatter(data[labels == 1, 0],data[labels == 1, 1], color="red")

# **From Clusters to Polygons**

Now, the points are clustered, but they need to be transformed into polygon for a shapefile output. It is done by adding cluster information into the pandas dataframe, before using it with geopandas to calculate a list of polygons using convex hull implementation in the library.

In [ ]:
dataset["cluster"] = pd.Series(labels)

In [ ]:
dataset.head()

# **Convert Pandas into Geopandas dataframe**

In [ ]:
hotspot_gdf = gpd.GeoDataFrame(
    dataset, geometry=gpd.points_from_xy(dataset.longitude, dataset.latitude)
    )

hotspot_gdf

Generate a list of polygons.

In [ ]:
polygon_list = []

for i in np.unique(labels):
  cluster_rows = hotspot_gdf[hotspot_gdf.cluster == i]
  poly = cluster_rows.unary_union.convex_hull
  polygon_list.append(poly)

In [ ]:
polygon_list

Append the list of polygons in GeoPandas and export it into a shapefile.

In [ ]:
multi_gdf = gpd.GeoDataFrame(geometry=polygon_list)
multi_gdf.to_file(filename=r'multi.shp', driver='ESRI Shapefile')

# **Functions**

In [ ]:
def to_map(df,zoom=6):
    """
    Display band on an interactive map.
    Description
    ----------
    Display xarray.Dataset in a folium map with multiple basemap options.
    Parameters
    ----------
    df: xarray.Dataset
        dataset with single time step, including bands "M13".
    Returns
    -------
    map: Folium Map
        Folium map with single band displayed as images with layer control.
    """

    #error catching
    assert isinstance(df, xr.Dataset),"Input has to be a xarray.Dataset."
    
    band = df.M13_log.values
      
    #boundary of the image on the map
    min_lon = df.lon.min().values.astype(float) + 0.0
    max_lon = df.lon.max().values.astype(float) + 0.0
    min_lat = df.lat.min().values.astype(float) + 0.0
    max_lat = df.lat.max().values.astype(float) + 0.0
    
    #create basemap for folium
    basemaps = {
        'Google Satellite': folium.TileLayer(
            tiles = 'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
            attr = 'Google',
            name = 'Basemap: Google Satellite',
            overlay = True,
            control = True
        ),
        'Google Terrain': folium.TileLayer(
            tiles = 'https://mt1.google.com/vt/lyrs=p&x={x}&y={y}&z={z}',
            attr = 'Google',
            name = 'Basemap: Google Terrain',
            overlay = True,
            control = True
        )
    }
    
    #display layers on map
    map_ = folium.Map(location=[(min_lat+max_lat)/2, (min_lon+max_lon)/2], zoom_start = zoom)
    basemaps['Google Satellite'].add_to(map_)
    basemaps['Google Terrain'].add_to(map_)

    factor = len(df.lat)*len(df.lon)/5000000
    img = cv.resize(np.flipud(np.fliplr(band.astype('float32'))), 
                         dsize=(math.ceil(len(df.lat)/factor), 
                                math.ceil(len(df.lon)/factor)), 
                         interpolation=cv.INTER_CUBIC)

    folium.raster_layers.ImageOverlay(
        img,[[min_lat, min_lon], [max_lat, max_lon]], name='image',colormap=cm.viridis
        ).add_to(map_)
  
    folium.LayerControl().add_to(map_)
    return map_